# Calibración, PR Curves y Fine-tuning

Calibración del modelo Atrato sobre datos de Colombia, selección de umbral operativo y preparación para fine-tuning.


In [ ]:
import sys, os, json, glob
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path('.').resolve()
sys.path.insert(0, str(ROOT / 'src'))
print('ROOT:', ROOT)


## 1. Cargar predicciones e inferencias previas

Cargamos las predicciones guardadas (media, máscara, entropía) y máscaras GT.


In [ ]:
UNCERT_DIR = ROOT / 'tmp_multisource_test' / 'uncertainty_full'
MS_MASK_DIR = ROOT / 'tmp_multisource_test' / 'out_masks_full'

def load_tif_band(path):
    arr = np.array(Image.open(path)).astype(np.float32)
    return arr

scenes = ['scene_001', 'scene_002']
all_probs = []
all_gts = []

for scene in scenes:
    mean_path = UNCERT_DIR / f'{scene}_mean.tif'
    mask_path = MS_MASK_DIR / f'{scene}.npy'
    
    if mean_path.exists() and mask_path.exists():
        pred_prob = load_tif_band(mean_path)
        gt_mask = np.load(mask_path).astype(np.float32)
        
        all_probs.append(pred_prob)
        all_gts.append(gt_mask)
        print(f'{scene}: pred shape {pred_prob.shape}, gt shape {gt_mask.shape}')

# Concatenar
all_probs_flat = np.concatenate([p.ravel() for p in all_probs])
all_gts_flat = np.concatenate([g.ravel() for g in all_gts])

print(f'\nTotal píxeles: {len(all_gts_flat):,}')
print(f'GT positivos: {np.sum(all_gts_flat > 0.5):,} ({100*np.mean(all_gts_flat>0.5):.2f}%)')
print(f'Predichos (p>0.5): {np.sum(all_probs_flat > 0.5):,}')


## 2. Curvas Precision-Recall y selección de umbral

Calculamos PR curve para todos los umbrales y hallamos puntos óptimos.


In [ ]:
def compute_pr_curve(gt, probs, n_thresholds=200):
    thresholds = np.linspace(0, 1, n_thresholds)
    precisions, recalls, f1s, ious = [], [], [], []
    gt_binary = (gt > 0.5).astype(np.float32)
    
    for thresh in thresholds:
        pred = (probs >= thresh).astype(np.float32)
        tp = np.sum(gt_binary * pred)
        fp = np.sum((1 - gt_binary) * pred)
        fn = np.sum(gt_binary * (1 - pred))
        tn = np.sum((1 - gt_binary) * (1 - pred))
        
        prec = tp / max(tp + fp, 1e-8)
        rec = tp / max(tp + fn, 1e-8)
        f1 = 2 * prec * rec / max(prec + rec, 1e-8)
        iou = tp / max(tp + fp + fn, 1e-8)
        
        precisions.append(prec)
        recalls.append(rec)
        f1s.append(f1)
        ious.append(iou)
    
    return thresholds, np.array(precisions), np.array(recalls), np.array(f1s), np.array(ious)

print('Computando PR curves...')
thresholds, precisions, recalls, f1s, ious = compute_pr_curve(
    all_gts_flat, all_probs_flat, n_thresholds=200
)
print('✓ PR curves computadas')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# PR curve
axes[0].plot(recalls, precisions, 'b-', linewidth=2)
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Curva Precision-Recall')
axes[0].grid(alpha=0.3)

# F1 vs threshold
best_f1_idx = np.argmax(f1s)
axes[1].plot(thresholds, f1s, 'g-', linewidth=2)
axes[1].plot(thresholds[best_f1_idx], f1s[best_f1_idx], 'ro', markersize=8,
             label=f'Max F1={f1s[best_f1_idx]:.3f}')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('F1-Score')
axes[1].set_title('F1 vs Threshold')
axes[1].grid(alpha=0.3)
axes[1].legend()

# IoU vs threshold
best_iou_idx = np.argmax(ious)
axes[2].plot(thresholds, ious, 'purple', linewidth=2)
axes[2].plot(thresholds[best_iou_idx], ious[best_iou_idx], 'ro', markersize=8,
             label=f'Max IoU={ious[best_iou_idx]:.3f}')
axes[2].set_xlabel('Threshold')
axes[2].set_ylabel('IoU')
axes[2].set_title('IoU vs Threshold')
axes[2].grid(alpha=0.3)
axes[2].legend()

fig.suptitle('Selección de umbral operativo', fontsize=13)
fig.tight_layout()
plt.show()


In [ ]:
targets = [0.70, 0.75, 0.80, 0.85, 0.90]
threshold_results = {}

print('='*70)
print('UMBRALES ÓPTIMOS POR OBJETIVO DE RECALL')
print('='*70)
print()

for target_recall in targets:
    valid_idx = np.where(recalls >= target_recall)[0]
    if len(valid_idx) > 0:
        best_idx = valid_idx[np.argmax(f1s[valid_idx])]
        t_opt = thresholds[best_idx]
        
        threshold_results[target_recall] = {
            'threshold': float(t_opt),
            'precision': float(precisions[best_idx]),
            'recall': float(recalls[best_idx]),
            'f1': float(f1s[best_idx]),
            'iou': float(ious[best_idx]),
        }
        
        print(f'Target Recall ≥ {target_recall:.0%}:')
        print(f'  Threshold: {t_opt:.4f}')
        print(f'  Precision: {precisions[best_idx]:.4f}')
        print(f'  Recall:    {recalls[best_idx]:.4f}')
        print(f'  F1:        {f1s[best_idx]:.4f}')
        print(f'  IoU:       {ious[best_idx]:.4f}')
    else:
        print(f'Target Recall ≥ {target_recall:.0%}: NO ALCANZABLE')
    print()


## 3. Preparación para Fine-tuning

Preparamos splits de datos y recomendaciones para fine-tuning corto.


In [ ]:
# Datos Colombia para fine-tuning
col_stacks = list((ROOT / 'data' / 'Colombia' / 'training' / 'stacks').glob('*.npy'))
col_masks = list((ROOT / 'data' / 'Colombia' / 'training' / 'masks').glob('*.npy'))

print(f'Tiles Colombia trovados: {len(col_stacks)}')

# Split 70-30
rng = np.random.default_rng(42)
n_total = len(col_stacks)
n_train = int(0.70 * n_total)
n_val = n_total - n_train

indices = rng.choice(n_total, n_total, replace=False)
train_idx = indices[:n_train]
val_idx = indices[n_train:]

print(f'\nFine-tuning split (70-30):')
print(f'  Train: {len(train_idx)} tiles')
print(f'  Val:   {len(val_idx)} tiles')
print()

# Guardar split
split_info = {
    'n_train': int(len(train_idx)),
    'n_val': int(len(val_idx)),
    'train_names': [Path(col_stacks[i]).stem for i in train_idx],
    'val_names': [Path(col_stacks[i]).stem for i in val_idx],
}

OUTPUT_DIR = ROOT / 'tmp_evaluation'
OUTPUT_DIR.mkdir(exist_ok=True)
split_file = OUTPUT_DIR / 'finetune_split.json'
with open(split_file, 'w') as f:
    json.dump(split_info, f, indent=2)
print(f'✓ Guardado: {split_file.name}')


In [ ]:
print('='*70)
print('RECOMENDACIONES PARA FINE-TUNING CORTO')
print('='*70)
print()
print('[PARÁMETROS ENTRENAMIENTO]')
print('  • Epochs: 10-20 (early stopping con patience=3)')
print('  • Learning rate: 1e-4 (reducido desde entrenamiento inicial 1e-3)')
print('  • Batch size: 4-8 (depende de RAM disponible)')
print('  • Optimizer: Adam')
print('  • Loss: WeightedBCE o Dice (para manejar desbalance de clases)')
print('  • Min-delta: 0.001')
print()
print('[DATOS]')
print(f'  • Tiles entrenamiento: {len(train_idx)}')
print(f'  • Tiles validación: {len(val_idx)}')
print('  • Augmentación: rotaciones 90°, flips, pequeños escalados')
print()
print('[UMBRAL OPERATIVO RECOMENDADO]')
if 0.80 in threshold_results:
    r = threshold_results[0.80]
    print(f'  • Para Recall ≥ 80%: threshold={r["threshold"]:.4f}')
    print(f'    → F1={r["f1"]:.4f}, Precision={r["precision"]:.4f}, IoU={r["iou"]:.4f}')
print()
print('[WORKFLOW]')
print('  1. Cargar checkpoint: data/atrato_training/checkpoints/atrato_unet.pt')
print('  2. Ejecutar fine-tuning 10-20 épocas con train_idx')
print('  3. Evaluar sobre val_idx con diferentes umbrales')
print('  4. Guardar nuevo checkpoint')
print('  5. Comparar métricas pre vs post fine-tuning')
